# Data Warehouse Development

## Project

India Job Market Intelligence Platform

---

## Objective

Design and implement a dimensional data warehouse from the cleaned job market dataset using MySQL.

The objective is to transform a denormalized transactional dataset into a Star Schema suitable for Business Intelligence reporting and Power BI dashboarding.

---

## Tools Used

- MySQL
- SQL
- Jupyter Notebook
- SQLAlchemy
- Pandas

---

## Deliverables

✔ Star Schema

✔ Dimension Tables

✔ Fact Table

✔ Primary Keys

✔ Foreign Keys

✔ SQL Validation

✔ Optimized Model for Power BI

# Star Schema Design

The analytical model consists of:

• Fact Table
    - fact_jobs

• Dimension Tables
    - dim_company
    - dim_location
    - dim_role
    - dim_salary
    - dim_experience

The grain of the fact table is:

> One row represents one unique job posting.

# Data Warehouse Development Process

The implementation follows the Kimball Dimensional Modeling methodology.

Steps:

1. Identify the business process

2. Define the grain

3. Identify dimensions

4. Identify facts

5. Create dimension tables

6. Populate dimensions

7. Create fact table

8. Create relationships

9. Validate the warehouse

In [1]:
# importing libraries
import pandas as pd

from sqlalchemy import create_engine
from sqlalchemy import text

In [2]:
# connect to mysql
MYSQL_USER = "root"
MYSQL_PASSWORD = "27104720A"

MYSQL_HOST = "localhost"

MYSQL_PORT = 3306

MYSQL_DATABASE = "job_market_analytics"

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

print("Database Connected Successfully")

Database Connected Successfully


In [5]:
# verify the connection
pd.read_sql(
"""
SHOW TABLES;
""",
engine
)

,Tables_in_job_market_analytics
0,dim_company
1,dim_location
2,dim_role
3,fact_jobs
4,jobs_final
5,jobs_raw
6,vw_job_overview


# Step 1

## Company Dimension

### Objective

Store company information only once and assign each company a surrogate key.

### Business Reason

Company information is repeated thousands of times in the raw dataset.

Creating a Company Dimension reduces redundancy and improves analytical performance.

In [13]:
pd.read_sql(
"""
SELECT
company_name,
company_rating,
company_rating_category,
company_size_bucket
FROM jobs_final
LIMIT 10;
""",
engine
)

,company_name,company_rating,company_rating_category,company_size_bucket
0,Cisco,4.1,Very Good,Large (1000+)
1,Caterpillar Inc,4.1,Very Good,Small/Startup (<100)
2,Foreign IT Consulting MNC,3.6,Good,Large (1000+)
3,Fortune 500 IT Services Company,3.6,Good,Small/Startup (<100)
4,Fortune 500 Product-based MNC,3.6,Good,Small/Startup (<100)
5,Top Rated IT Enterprise,3.6,Good,Small/Startup (<100)
6,Leading IT Firm,3.6,Good,Small/Startup (<100)
7,Exxon Mobil Corporation,3.7,Good,Small/Startup (<100)
8,Google,4.4,Very Good,Large (1000+)
9,Hsbc,3.8,Good,Large (1000+)


# Count Unique Companies

In [16]:
pd.read_sql(
"""
SELECT COUNT(DISTINCT company_name)
AS Total_Companies
FROM jobs_final;
""",
engine
)

,Total_Companies
0,6944


## SQL Implementation

The following SQL statement creates the Company Dimension using a surrogate primary key.

# Create Table

In [20]:
query = """

DROP TABLE IF EXISTS dim_company;

CREATE TABLE dim_company(

company_key INT AUTO_INCREMENT PRIMARY KEY,

company_name VARCHAR(255),

company_rating DECIMAL(3,1),

company_rating_category VARCHAR(50),

company_size_bucket VARCHAR(100)

);

"""

with engine.begin() as conn:

    for statement in query.split(";"):

        if statement.strip():

            conn.execute(text(statement))

# Populate Company Dimension(feed data into the dimension table)

In [23]:
query = """

INSERT INTO dim_company(

company_name,

company_rating,

company_rating_category,

company_size_bucket

)

SELECT DISTINCT

company_name,

company_rating,

company_rating_category,

company_size_bucket

FROM jobs_final;

"""

with engine.begin() as conn:

    conn.execute(text(query))

## Validation

Now, The Company Dimension should contain one record per unique company.

In [26]:
pd.read_sql(
"""
SELECT COUNT(*)
AS Total_Companies
FROM dim_company;
""",
engine
)

,Total_Companies
0,7022


# Preview of the dimension table dim_company

In [29]:
pd.read_sql(
"""
SELECT *
FROM dim_company
LIMIT 20;
""",
engine
)

,company_key,company_name,company_rating,company_rating_category,company_size_bucket
0,1,Cisco,4.1,Very Good,Large (1000+)
1,2,Caterpillar Inc,4.1,Very Good,Small/Startup (<100)
2,3,Foreign IT Consulting MNC,3.6,Good,Large (1000+)
3,4,Fortune 500 IT Services Company,3.6,Good,Small/Startup (<100)
4,5,Fortune 500 Product-based MNC,3.6,Good,Small/Startup (<100)
5,6,Top Rated IT Enterprise,3.6,Good,Small/Startup (<100)
6,7,Leading IT Firm,3.6,Good,Small/Startup (<100)
7,8,Exxon Mobil Corporation,3.7,Good,Small/Startup (<100)
8,9,Google,4.4,Very Good,Large (1000+)
9,10,Hsbc,3.8,Good,Large (1000+)


## Implementation Summary

The Company Dimension was successfully created and populated.

Key accomplishments:

- Generated surrogate primary keys.
- Removed duplicate company records.
- Created a reusable dimension for future joins.
- Improved the dimensional design of the analytical model.

# Location Dimension Design & Implementation

# Phase 2

## Location Dimension

### Objective

Create a reusable Location Dimension by extracting unique locations from the job postings.

### Business Purpose

Location is a descriptive attribute frequently used for geographical analysis, regional hiring trends, and dashboard filtering.

# Step 1: Understand the Source Data

In [37]:
pd.read_sql("""
SELECT
    location,
    scraped_city,
    work_mode
FROM jobs_final
LIMIT 20;
""", engine)

,location,scraped_city,work_mode
0,Bengaluru,Bangalore,On-site
1,Bengaluru,Bangalore,On-site
2,"Hyderabad, Chennai, Bengaluru",Bangalore,On-site
3,"Mumbai, Bengaluru",Bangalore,On-site
4,"Pune, Bengaluru",Bangalore,On-site
5,"Chennai, Bengaluru",Bangalore,On-site
6,"Chennai, Bengaluru",Bangalore,On-site
7,Bengaluru,Bangalore,On-site
8,Bengaluru,Bangalore,On-site
9,Bengaluru,Bangalore,On-site


# Step 2: Investigate the Data

In [40]:
pd.read_sql("""
SELECT COUNT(DISTINCT location) AS unique_locations
FROM jobs_final;
""", engine)

,unique_locations
0,1563


In [42]:
pd.read_sql("""
SELECT location, COUNT(*) AS job_count
FROM jobs_final
GROUP BY location
ORDER BY job_count DESC
LIMIT 20;
""", engine)

,location,job_count
0,Bengaluru,2503
1,Remote,2003
2,Gurugram,1886
3,Noida,1822
4,Mumbai,1748
5,Pune,1440
6,Chennai,1363
7,Ahmedabad,1098
8,Kolkata,765
9,"Kolkata, Mumbai, New Delhi, Hyderabad, Pune, C...",647


# Step 3: Let's Inspect the Data First

In [44]:
pd.read_sql("""
DESCRIBE jobs_final;
""", engine)

,Field,Type,Null,Key,Default,Extra
0,job_id,bigint,YES,,None,
1,job_title,text,YES,,None,
2,company_name,text,YES,,None,
3,company_rating,double,YES,,None,
4,location,text,YES,,None,
5,scraped_city,text,YES,,None,
6,role_category,text,YES,,None,
7,experience_raw,text,YES,,None,
8,experience_min_yrs,bigint,YES,,None,
9,experience_max_yrs,bigint,YES,,None,


In [47]:
pd.read_sql("""
SELECT
    location,
    scraped_city
FROM jobs_final
LIMIT 20;
""", engine)

,location,scraped_city
0,Bengaluru,Bangalore
1,Bengaluru,Bangalore
2,"Hyderabad, Chennai, Bengaluru",Bangalore
3,"Mumbai, Bengaluru",Bangalore
4,"Pune, Bengaluru",Bangalore
5,"Chennai, Bengaluru",Bangalore
6,"Chennai, Bengaluru",Bangalore
7,Bengaluru,Bangalore
8,Bengaluru,Bangalore
9,Bengaluru,Bangalore


### Design Consideration

The original `location` field contains both single-city and multi-city job postings (e.g., "Mumbai, Bengaluru" or "Hybrid - Gurugram, Bengaluru"), making it a non-atomic attribute.

For enterprise-scale implementations, this would typically be modeled using a bridge table (`bridge_job_location`) to support many-to-many relationships between jobs and cities.

To keep the dimensional model appropriate for this portfolio project while preserving analytical value, a simplified `dim_location` will be created using the original `location` text together with the engineered `primary_city`. This approach supports common reporting scenarios without introducing unnecessary modeling complexity.

# Phase 2.1 – Data Exploration

# Step 1 – Count Unique Locations

In [52]:
pd.read_sql("""
SELECT COUNT(DISTINCT location) AS total_locations
FROM jobs_final;
""", engine)

,total_locations
0,1563


# Step 2 – Top 20 Locations

In [55]:
pd.read_sql("""
SELECT
    location,
    COUNT(*) AS total_jobs
FROM jobs_final
GROUP BY location
ORDER BY total_jobs DESC
LIMIT 20;
""", engine)

,location,total_jobs
0,Bengaluru,2503
1,Remote,2003
2,Gurugram,1886
3,Noida,1822
4,Mumbai,1748
5,Pune,1440
6,Chennai,1363
7,Ahmedabad,1098
8,Kolkata,765
9,"Kolkata, Mumbai, New Delhi, Hyderabad, Pune, C...",647


# Step 3 – Count Primary Cities

In [58]:
pd.read_sql("""
SELECT
    primary_city,
    COUNT(*) AS total_jobs
FROM jobs_final
GROUP BY primary_city
ORDER BY total_jobs DESC;
""", engine)

,primary_city,total_jobs
0,Mumbai,3276
1,Bangalore,2830
2,Chennai,2666
3,Pune,2606
4,Noida,2585
...,...,...
193,Panchkula,1
194,Mumbai(Mahalaxmi),1
195,Mumbai(Mulund),1
196,Mumbai(Goregaon East +3),1


# Step 4 – Multi-location Jobs

In [61]:
pd.read_sql("""
SELECT
    is_multi_location,
    COUNT(*) AS total_jobs
FROM jobs_final
GROUP BY is_multi_location;
""", engine)

,is_multi_location,total_jobs
0,0,16746
1,1,6455


# Phase 2: Location Dimension Design & Implementation

## Objective

Design a reusable Location Dimension by extracting unique geographical attributes from the cleaned job postings dataset.

## Business Rationale

The original `location` field contains both single-city and multi-city values, making it non-atomic. To improve analytical usability while maintaining simplicity, the dimension retains the original location string, an engineered `primary_city`, and a flag indicating whether the posting is available across multiple locations.

This design supports common reporting scenarios such as:

- Hiring by City
- Single vs Multi-location Jobs
- Geographic Distribution
- Dashboard Filtering

while avoiding unnecessary complexity such as bridge tables.

# Step 1 – Create Dimension Table

In [66]:
query = """
DROP TABLE IF EXISTS dim_location;

CREATE TABLE dim_location (

    location_key INT AUTO_INCREMENT PRIMARY KEY,

    location VARCHAR(255),

    primary_city VARCHAR(100),

    is_multi_location BOOLEAN

);
"""

with engine.begin() as conn:
    for statement in query.split(";"):
        if statement.strip():
            conn.execute(text(statement))

print("dim_location created successfully.")

dim_location created successfully.


# Step 2 – Populate Table

In [69]:
query = """
INSERT INTO dim_location
(
    location,
    primary_city,
    is_multi_location
)

SELECT DISTINCT

    location,
    primary_city,
    is_multi_location

FROM jobs_final;
"""

with engine.begin() as conn:
    conn.execute(text(query))

print("Data inserted successfully.")

Data inserted successfully.


# Step 3 – Validation

In [72]:
pd.read_sql("""
SELECT COUNT(*)
FROM dim_location;
""", engine)

,COUNT(*)
0,1563


# Step 4 – Preview

In [75]:
pd.read_sql("""
SELECT *
FROM dim_location
LIMIT 20;
""", engine)

,location_key,location,primary_city,is_multi_location
0,1,Bengaluru,Bangalore,0
1,2,"Hyderabad, Chennai, Bengaluru",Hyderabad,1
2,3,"Mumbai, Bengaluru",Mumbai,1
3,4,"Pune, Bengaluru",Pune,1
4,5,"Chennai, Bengaluru",Chennai,1
5,6,"Hybrid - Gurugram, Bengaluru",Gurgaon,1
6,7,"Hybrid - Hyderabad, Gurugram, Bengaluru",Hyderabad,1
7,8,"Hybrid - Noida, Kolkata, Bengaluru",Noida,1
8,9,Bangalore Rural,Bangalore,0
9,10,Bengaluru(Old Airport Road),Bangalore,0


# Phase 3: Role Dimension Design & Implementation

## Objective

Design a Role Dimension to categorize job postings by title, role category, skill domain, seniority level, work arrangement, and fresher eligibility.

## Business Rationale

Job-related attributes are frequently used for filtering and segmentation in recruitment analytics. Creating a dedicated Role Dimension reduces redundancy and supports flexible reporting across job functions, experience levels, and hiring trends.

# Step 1: Explore the Source Data

In [79]:
pd.read_sql("""
SELECT
    job_title,
    role_category,
    skill_domain,
    work_mode,
    is_senior,
    is_fresher_friendly
FROM jobs_final
LIMIT 20;
""", engine)

,job_title,role_category,skill_domain,work_mode,is_senior,is_fresher_friendly
0,Data Scientist,Data Scientist,Business Intelligence,On-site,0,0
1,Data Scientist,Data Scientist,AI/ML/DL,On-site,0,0
2,Analytics Data Scientist,Data Scientist,Business Intelligence,On-site,0,0
3,Data Scientist,Data Scientist,Data Science,On-site,0,1
4,Sr. Artificial Intelligence Engineer,Data Scientist,Data Science,On-site,1,0
5,Data Scientist,Data Scientist,Data Science,On-site,0,0
6,Data Scientist,Data Scientist,Data Science,On-site,0,0
7,Advanced Data Scientist,Data Scientist,AI/ML/DL,On-site,0,0
8,"Data Scientist, Google Play, Product",Data Scientist,AI/ML/DL,On-site,0,0
9,Machine Learning Data Scientist,Data Scientist,AI/ML/DL,On-site,0,0


# Step 2: Check Unique Values

In [82]:
pd.read_sql("""
SELECT
    COUNT(DISTINCT job_title) AS unique_job_titles,
    COUNT(DISTINCT role_category) AS unique_role_categories,
    COUNT(DISTINCT skill_domain) AS unique_skill_domains,
    COUNT(DISTINCT work_mode) AS unique_work_modes
FROM jobs_final;
""", engine)

,unique_job_titles,unique_role_categories,unique_skill_domains,unique_work_modes
0,12888,6,5,3


# Step 3: Check Work Mode Distribution

In [85]:
pd.read_sql("""
SELECT
    work_mode,
    COUNT(*) AS total_jobs
FROM jobs_final
GROUP BY work_mode
ORDER BY total_jobs DESC;
""", engine)

,work_mode,total_jobs
0,On-site,18701
1,Hybrid,2347
2,Remote,2153


# Step 4: Check Role Categories

In [88]:
pd.read_sql("""
SELECT
    role_category,
    COUNT(*) AS total_jobs
FROM jobs_final
GROUP BY role_category
ORDER BY total_jobs DESC;
""", engine)

,role_category,total_jobs
0,Data Scientist,6455
1,Data Analyst,4729
2,Business Analyst,4505
3,Machine Learning Engineer,4004
4,Data Engineer,1922
5,Python Developer,1586


# Step 5: Check Skill Domains

In [93]:
pd.read_sql("""
SELECT
    skill_domain,
    COUNT(*) AS total_jobs
FROM jobs_final
GROUP BY skill_domain
ORDER BY total_jobs DESC;
""", engine)

,skill_domain,total_jobs
0,Business Intelligence,11124
1,AI/ML/DL,6591
2,Cloud & DevOps,2344
3,Data Engineering,1595
4,Data Science,1547


# Phase 3: Role Dimension Design

## Design Decision

The Role Dimension consolidates descriptive job attributes that are frequently used for business reporting and dashboard filtering.

Instead of storing these attributes repeatedly in the Fact Table, they are maintained in a dedicated dimension.

This enables analyses such as:

- Hiring by Role
- Hiring by Domain
- Remote vs Hybrid Jobs
- Fresher-Friendly Opportunities
- Senior-Level Hiring
- Experience Tier Distribution

# Step 1 – Create Table

In [97]:
query = """
DROP TABLE IF EXISTS dim_role;

CREATE TABLE dim_role (

    role_key INT AUTO_INCREMENT PRIMARY KEY,

    job_title TEXT,

    role_category VARCHAR(100),

    skill_domain VARCHAR(100),

    work_mode VARCHAR(50),

    experience_tier VARCHAR(50),

    is_senior BOOLEAN,

    is_fresher_friendly BOOLEAN

);
"""

with engine.begin() as conn:
    for statement in query.split(";"):
        if statement.strip():
            conn.execute(text(statement))

print("dim_role created successfully.")

dim_role created successfully.


# Step 2 – Populate Table

In [100]:
query = """
INSERT INTO dim_role
(
    job_title,
    role_category,
    skill_domain,
    work_mode,
    experience_tier,
    is_senior,
    is_fresher_friendly
)

SELECT DISTINCT

    job_title,
    role_category,
    skill_domain,
    work_mode,
    experience_tier,
    is_senior,
    is_fresher_friendly

FROM jobs_final;
"""

with engine.begin() as conn:
    conn.execute(text(query))

print("Data inserted successfully.")

Data inserted successfully.


# Step 3 – Validation

In [104]:
pd.read_sql("""
SELECT COUNT(*) AS total_roles
FROM dim_role;
""", engine)

,total_roles
0,16181


# Step 4 – Preview

In [107]:
pd.read_sql("""
SELECT *
FROM dim_role
LIMIT 20;
""", engine)

,role_key,job_title,role_category,skill_domain,work_mode,experience_tier,is_senior,is_fresher_friendly
0,1,Data Scientist,Data Scientist,Business Intelligence,On-site,Senior (6-8 Yrs),0,0
1,2,Data Scientist,Data Scientist,AI/ML/DL,On-site,Mid (3-5 Yrs),0,0
2,3,Analytics Data Scientist,Data Scientist,Business Intelligence,On-site,Mid (3-5 Yrs),0,0
3,4,Data Scientist,Data Scientist,Data Science,On-site,Fresher,0,1
4,5,Sr. Artificial Intelligence Engineer,Data Scientist,Data Science,On-site,Mid (3-5 Yrs),1,0
5,6,Data Scientist,Data Scientist,Data Science,On-site,Mid (3-5 Yrs),0,0
6,7,Advanced Data Scientist,Data Scientist,AI/ML/DL,On-site,Mid (3-5 Yrs),0,0
7,8,"Data Scientist, Google Play, Product",Data Scientist,AI/ML/DL,On-site,Mid (3-5 Yrs),0,0
8,9,Machine Learning Data Scientist,Data Scientist,AI/ML/DL,On-site,Mid (3-5 Yrs),0,0
9,10,Sr Data Scientist - Advanced Machine Learning,Data Scientist,AI/ML/DL,On-site,Mid (3-5 Yrs),0,0


## Architect's Reflection

### Why was `job_title` included despite having over 12,000 unique values?

Although `job_title` has high cardinality, it is a descriptive attribute rather than a measurable fact. Business users frequently search, filter, and analyze hiring trends using job titles, making it appropriate for a dimension.

### Why was `experience_tier` included?

Unlike numeric experience values, `experience_tier` is a business-friendly classification (e.g., Entry, Mid, Senior) that supports reporting and dashboard segmentation.

### Why were numeric experience and salary values excluded?

Continuous numerical measures are better stored in the Fact Table, where they can be aggregated efficiently for analytical reporting.

# Phase 4: Dimension Key Lookup

## Objective

Before constructing the Fact Table, each job posting must be linked to the corresponding dimension records.

This process replaces descriptive business attributes with surrogate keys, ensuring referential integrity and reducing data redundancy.

The lookup process simulates a standard ETL workflow used in dimensional data warehouses.

# Step 1 – Verify Company Mapping

# Before joining, let's make sure every company exists in dim_company

In [112]:
pd.read_sql("""
SELECT COUNT(*) AS unmatched_companies
FROM jobs_final j
LEFT JOIN dim_company c
ON j.company_name = c.company_name
AND j.company_rating = c.company_rating
AND j.company_rating_category = c.company_rating_category
AND j.company_size_bucket = c.company_size_bucket
WHERE c.company_key IS NULL;
""", engine)

,unmatched_companies
0,0


# Step 2 – Verify Location Mapping

In [115]:
pd.read_sql("""
SELECT COUNT(*) AS unmatched_locations
FROM jobs_final j
LEFT JOIN dim_location l
ON j.location = l.location
AND j.primary_city = l.primary_city
AND j.is_multi_location = l.is_multi_location
WHERE l.location_key IS NULL;
""", engine)

,unmatched_locations
0,0


# Step 3 – Verify Role Mapping

In [118]:
pd.read_sql("""
SELECT COUNT(*) AS unmatched_roles
FROM jobs_final j
LEFT JOIN dim_role r
ON j.job_title = r.job_title
AND j.role_category = r.role_category
AND j.skill_domain = r.skill_domain
AND j.work_mode = r.work_mode
AND j.experience_tier = r.experience_tier
AND j.is_senior = r.is_senior
AND j.is_fresher_friendly = r.is_fresher_friendly
WHERE r.role_key IS NULL;
""", engine)

,unmatched_roles
0,0


## Quality Assurance

Before loading the Fact Table, lookup validation was performed to ensure that every record in the cleaned dataset had a corresponding record in each dimension table.

Successful validation confirms that the dimensional model maintains referential integrity.

# Phase 5: Fact Table Design & Implementation

## Objective

Construct the central Fact Table by replacing descriptive attributes with surrogate keys from the dimension tables.

The Fact Table stores measurable business metrics while maintaining relationships with the Company, Location, and Role dimensions.

## Business Benefits

- Eliminates redundancy
- Improves query performance
- Supports analytical reporting
- Enables Power BI star schema
- Maintains referential integrity

# Step 1 – Create the Table

In [129]:
query = """
DROP TABLE IF EXISTS fact_jobs;

CREATE TABLE fact_jobs (

    fact_key INT AUTO_INCREMENT PRIMARY KEY,

    job_id BIGINT,

    company_key INT,

    location_key INT,

    role_key INT,

    salary_min_lpa DOUBLE,

    salary_max_lpa DOUBLE,

    salary_midpoint_lpa DOUBLE,

    experience_min_yrs INT,

    experience_max_yrs INT,

    experience_span INT,

    skills_count INT,

    days_since_posted INT,

    salary_disclosed BOOLEAN,

    salary_negotiable BOOLEAN,

    FOREIGN KEY (company_key)
        REFERENCES dim_company(company_key),

    FOREIGN KEY (location_key)
        REFERENCES dim_location(location_key),

    FOREIGN KEY (role_key)
        REFERENCES dim_role(role_key)

);
"""

with engine.begin() as conn:
    for statement in query.split(";"):
        if statement.strip():
            conn.execute(text(statement))

print("fact_jobs created successfully.")

fact_jobs created successfully.


# Phase 5.1: ETL Lookup Pipeline

## Objective

Transform the cleaned transactional dataset into a warehouse-ready dataset by replacing descriptive attributes with surrogate keys from each dimension table.

This step simulates the Lookup Transformation commonly used in ETL tools such as SSIS, Informatica, Talend, and Azure Data Factory.

The resulting dataset will be used to populate the Fact Table.

# Step 1 — Join Company Dimension

In [133]:
company_lookup = pd.read_sql("""

SELECT

    j.*,

    c.company_key

FROM jobs_final j

LEFT JOIN dim_company c

ON j.company_name = c.company_name
AND j.company_rating = c.company_rating
AND j.company_rating_category = c.company_rating_category
AND j.company_size_bucket = c.company_size_bucket;

""", engine)

company_lookup.head()

,job_id,job_title,company_name,company_rating,location,scraped_city,role_category,experience_raw,experience_min_yrs,experience_max_yrs,...,salary_negotiable,salary_range,experience_span,experience_data_valid,experience_issue,experience_midpoint,job_age_category,is_multi_location,company_rating_category,company_key
0,1,Data Scientist,Cisco,4.1,Bengaluru,Bangalore,Data Scientist,7-10 Yrs,7,10,...,0,Not Disclosed,3,1,Valid,8.5,Active (8–30 Days),0,Very Good,1
1,2,Data Scientist,Caterpillar Inc,4.1,Bengaluru,Bangalore,Data Scientist,4-8 Yrs,4,8,...,0,Not Disclosed,4,1,Valid,6.0,Active (8–30 Days),0,Very Good,2
2,3,Analytics Data Scientist,Foreign IT Consulting MNC,3.6,"Hyderabad, Chennai, Bengaluru",Bangalore,Data Scientist,4-9 Yrs,4,9,...,0,Not Disclosed,5,1,Valid,6.5,Active (8–30 Days),1,Good,3
3,4,Data Scientist,Fortune 500 IT Services Company,3.6,"Mumbai, Bengaluru",Bangalore,Data Scientist,10-14 Yrs,0,0,...,0,Not Disclosed,0,1,Valid,0.0,Active (8–30 Days),1,Good,4
4,5,Sr. Artificial Intelligence Engineer,Fortune 500 Product-based MNC,3.6,"Pune, Bengaluru",Bangalore,Data Scientist,5-10 Yrs,5,10,...,0,Not Disclosed,5,1,Valid,7.5,Active (8–30 Days),1,Good,5


In [135]:
# validation
company_lookup.shape

(23201, 41)

# Check Missing Keys

In [138]:
company_lookup['company_key'].isnull().sum()

0

# Step 2 — Join Location Dimension

In [141]:
location_lookup = pd.read_sql("""

SELECT

    j.*,

    l.location_key

FROM jobs_final j

LEFT JOIN dim_location l

ON j.location = l.location
AND j.primary_city = l.primary_city
AND j.is_multi_location = l.is_multi_location;

""", engine)

location_lookup.head()

,job_id,job_title,company_name,company_rating,location,scraped_city,role_category,experience_raw,experience_min_yrs,experience_max_yrs,...,salary_negotiable,salary_range,experience_span,experience_data_valid,experience_issue,experience_midpoint,job_age_category,is_multi_location,company_rating_category,location_key
0,1,Data Scientist,Cisco,4.1,Bengaluru,Bangalore,Data Scientist,7-10 Yrs,7,10,...,0,Not Disclosed,3,1,Valid,8.5,Active (8–30 Days),0,Very Good,1
1,2,Data Scientist,Caterpillar Inc,4.1,Bengaluru,Bangalore,Data Scientist,4-8 Yrs,4,8,...,0,Not Disclosed,4,1,Valid,6.0,Active (8–30 Days),0,Very Good,1
2,3,Analytics Data Scientist,Foreign IT Consulting MNC,3.6,"Hyderabad, Chennai, Bengaluru",Bangalore,Data Scientist,4-9 Yrs,4,9,...,0,Not Disclosed,5,1,Valid,6.5,Active (8–30 Days),1,Good,2
3,4,Data Scientist,Fortune 500 IT Services Company,3.6,"Mumbai, Bengaluru",Bangalore,Data Scientist,10-14 Yrs,0,0,...,0,Not Disclosed,0,1,Valid,0.0,Active (8–30 Days),1,Good,3
4,5,Sr. Artificial Intelligence Engineer,Fortune 500 Product-based MNC,3.6,"Pune, Bengaluru",Bangalore,Data Scientist,5-10 Yrs,5,10,...,0,Not Disclosed,5,1,Valid,7.5,Active (8–30 Days),1,Good,4


In [143]:
# validation
location_lookup.shape

(23201, 41)

# Missing Keys

In [146]:
location_lookup['location_key'].isnull().sum()

0

# Step 3 — Join Role Dimension

In [149]:
role_lookup = pd.read_sql("""

SELECT

    j.*,

    r.role_key

FROM jobs_final j

LEFT JOIN dim_role r

ON j.job_title = r.job_title
AND j.role_category = r.role_category
AND j.skill_domain = r.skill_domain
AND j.work_mode = r.work_mode
AND j.experience_tier = r.experience_tier
AND j.is_senior = r.is_senior
AND j.is_fresher_friendly = r.is_fresher_friendly;

""", engine)

role_lookup.head()

,job_id,job_title,company_name,company_rating,location,scraped_city,role_category,experience_raw,experience_min_yrs,experience_max_yrs,...,salary_negotiable,salary_range,experience_span,experience_data_valid,experience_issue,experience_midpoint,job_age_category,is_multi_location,company_rating_category,role_key
0,1,Data Scientist,Cisco,4.1,Bengaluru,Bangalore,Data Scientist,7-10 Yrs,7,10,...,0,Not Disclosed,3,1,Valid,8.5,Active (8–30 Days),0,Very Good,1
1,2,Data Scientist,Caterpillar Inc,4.1,Bengaluru,Bangalore,Data Scientist,4-8 Yrs,4,8,...,0,Not Disclosed,4,1,Valid,6.0,Active (8–30 Days),0,Very Good,2
2,3,Analytics Data Scientist,Foreign IT Consulting MNC,3.6,"Hyderabad, Chennai, Bengaluru",Bangalore,Data Scientist,4-9 Yrs,4,9,...,0,Not Disclosed,5,1,Valid,6.5,Active (8–30 Days),1,Good,3
3,4,Data Scientist,Fortune 500 IT Services Company,3.6,"Mumbai, Bengaluru",Bangalore,Data Scientist,10-14 Yrs,0,0,...,0,Not Disclosed,0,1,Valid,0.0,Active (8–30 Days),1,Good,4
4,5,Sr. Artificial Intelligence Engineer,Fortune 500 Product-based MNC,3.6,"Pune, Bengaluru",Bangalore,Data Scientist,5-10 Yrs,5,10,...,0,Not Disclosed,5,1,Valid,7.5,Active (8–30 Days),1,Good,5


In [151]:
# Validation
role_lookup.shape

(23201, 41)

# Missing Keys

In [154]:
role_lookup['role_key'].isnull().sum()

0

# Phase 5.2: Fact Table Population

## Objective

Populate the Fact Table by replacing descriptive business attributes with surrogate keys from the Company, Location, and Role dimensions.

This step completes the ETL process by loading warehouse-ready transactional data into the central Fact Table.

The resulting star schema enables efficient analytical queries and Power BI reporting.

# Step 1: Populate the Fact Table

In [159]:
from sqlalchemy import text

query = """
INSERT INTO fact_jobs
(
    job_id,
    company_key,
    location_key,
    role_key,
    salary_min_lpa,
    salary_max_lpa,
    salary_midpoint_lpa,
    experience_min_yrs,
    experience_max_yrs,
    experience_span,
    skills_count,
    days_since_posted,
    salary_disclosed,
    salary_negotiable
)

SELECT

    j.job_id,

    c.company_key,

    l.location_key,

    r.role_key,

    j.salary_min_lpa,

    j.salary_max_lpa,

    j.salary_midpoint_lpa,

    j.experience_min_yrs,

    j.experience_max_yrs,

    j.experience_span,

    j.skills_count,

    j.days_since_posted,

    j.salary_disclosed,

    j.salary_negotiable

FROM jobs_final j

INNER JOIN dim_company c

ON j.company_name = c.company_name
AND j.company_rating = c.company_rating
AND j.company_rating_category = c.company_rating_category
AND j.company_size_bucket = c.company_size_bucket

INNER JOIN dim_location l

ON j.location = l.location
AND j.primary_city = l.primary_city
AND j.is_multi_location = l.is_multi_location

INNER JOIN dim_role r

ON j.job_title = r.job_title
AND j.role_category = r.role_category
AND j.skill_domain = r.skill_domain
AND j.work_mode = r.work_mode
AND j.experience_tier = r.experience_tier
AND j.is_senior = r.is_senior
AND j.is_fresher_friendly = r.is_fresher_friendly;
"""

with engine.begin() as conn:
    conn.execute(text(query))

print("Fact table populated successfully.")

Fact table populated successfully.


# Step 2: Validate the Load

In [162]:
pd.read_sql("""
SELECT COUNT(*) AS total_records
FROM fact_jobs;
""", engine)

,total_records
0,23201


# Check for NULL Foreign Keys

In [165]:
pd.read_sql("""
SELECT

SUM(company_key IS NULL) AS company_nulls,

SUM(location_key IS NULL) AS location_nulls,

SUM(role_key IS NULL) AS role_nulls

FROM fact_jobs;
""", engine)

,company_nulls,location_nulls,role_nulls
0,0.0,0.0,0.0


# Check Duplicate Job IDs

In [168]:
pd.read_sql("""
SELECT

COUNT(*) AS total_rows,

COUNT(DISTINCT job_id) AS unique_jobs

FROM fact_jobs;
""", engine)

,total_rows,unique_jobs
0,23201,23201


# Preview the Fact Table

In [171]:
pd.read_sql("""
SELECT *
FROM fact_jobs
LIMIT 20;
""", engine)

,fact_key,job_id,company_key,location_key,role_key,salary_min_lpa,salary_max_lpa,salary_midpoint_lpa,experience_min_yrs,experience_max_yrs,experience_span,skills_count,days_since_posted,salary_disclosed,salary_negotiable
0,1,338,1,1,169,0.0,0.0,0.0,4,9,5,8,21,0,0
1,2,1,1,1,1,0.0,0.0,0.0,7,10,3,8,21,0,0
2,3,411,2,1,161,0.0,0.0,0.0,8,13,5,8,21,0,0
3,4,2,2,1,2,0.0,0.0,0.0,4,8,4,8,21,0,0
4,5,3,3,2,3,0.0,0.0,0.0,4,9,5,0,21,0,0
5,6,4,4,3,4,0.0,0.0,0.0,0,0,0,0,21,0,0
6,7,5,5,4,5,0.0,0.0,0.0,5,10,5,0,21,0,0
7,8,6,6,5,6,0.0,0.0,0.0,3,8,5,0,21,0,0
8,9,7,7,5,6,0.0,0.0,0.0,5,10,5,0,21,0,0
9,10,8,8,1,7,0.0,0.0,0.0,5,10,5,8,21,0,0


## Architect's Reflection

The Fact Table serves as the central repository of measurable business events.

Instead of storing repetitive descriptive information such as company names, locations, and job titles, surrogate keys are used to reference dimension tables.

This design:

- Reduces redundancy
- Improves storage efficiency
- Ensures referential integrity
- Simplifies Power BI relationships
- Supports scalable analytical queries

The ETL pipeline included lookup validation before loading, ensuring every job posting successfully mapped to the corresponding dimension records.

# Phase 6: Data Warehouse Quality Assurance

## Objective

Validate the integrity, completeness, and consistency of the dimensional data warehouse after the ETL process.

The purpose of this phase is to ensure that all dimension lookups, foreign key relationships, and business measures have been loaded correctly before analytical reporting begins.

# QA Check 1: Row Count Reconciliation

In [176]:
pd.read_sql("""
SELECT
    (SELECT COUNT(*) FROM jobs_final) AS source_rows,
    (SELECT COUNT(*) FROM fact_jobs) AS fact_rows;
""", engine)

,source_rows,fact_rows
0,23201,23201


# QA Check 2: Foreign Key Validation

In [187]:
pd.read_sql("""
SELECT

SUM(company_key IS NULL) AS company_nulls,

SUM(location_key IS NULL) AS location_nulls,

SUM(role_key IS NULL) AS role_nulls

FROM fact_jobs;
""", engine)

,company_nulls,location_nulls,role_nulls
0,0.0,0.0,0.0


# QA Check 3: Duplicate Business Keys

In [182]:
pd.read_sql("""
SELECT

COUNT(*) AS total_rows,

COUNT(DISTINCT job_id) AS unique_jobs

FROM fact_jobs;
""", engine)

,total_rows,unique_jobs
0,23201,23201


# QA Check 4: Orphan Records

In [185]:
pd.read_sql("""
SELECT COUNT(*) AS orphan_company_keys
FROM fact_jobs f
LEFT JOIN dim_company c
ON f.company_key = c.company_key
WHERE c.company_key IS NULL;
""", engine)

,orphan_company_keys
0,0


# QA Check 5: Salary Validation

In [190]:
pd.read_sql("""
SELECT

MIN(salary_min_lpa) AS minimum_salary,

MAX(salary_max_lpa) AS maximum_salary,

AVG(salary_midpoint_lpa) AS average_salary

FROM fact_jobs;
""", engine)

,minimum_salary,maximum_salary,average_salary
0,0.0,90.0,1.827872


# QA Check 6: Experience Validation

In [193]:
pd.read_sql("""
SELECT

MIN(experience_min_yrs),

MAX(experience_max_yrs),

AVG(experience_span)

FROM fact_jobs;
""", engine)

,MIN(experience_min_yrs),MAX(experience_max_yrs),AVG(experience_span)
0,0,31,3.694


# QA Check 7: Star Schema Overview

In [196]:
pd.read_sql("""
SELECT

(SELECT COUNT(*) FROM dim_company) AS companies,

(SELECT COUNT(*) FROM dim_location) AS locations,

(SELECT COUNT(*) FROM dim_role) AS roles,

(SELECT COUNT(*) FROM fact_jobs) AS jobs;
""", engine)

,companies,locations,roles,jobs
0,7022,1563,16181,23201


## Warehouse Validation Summary

The dimensional warehouse successfully passed all post-load quality assurance checks.

Validation confirmed:

- Complete ETL load
- Referential integrity
- No orphan foreign keys
- No duplicate business keys
- Consistent analytical measures
- Production-ready Star Schema

These validation steps mirror standard data warehouse QA practices used before publishing datasets for Business Intelligence reporting.

## Top Paying Compnies Power BI visual checking

In [7]:
pd.read_sql("""
SELECT
    c.company_name,
    COUNT(f.salary_max_lpa) AS disclosed_jobs,
    ROUND(AVG(f.salary_max_lpa), 2) AS avg_salary
FROM fact_jobs f
JOIN dim_company c
    ON f.company_key = c.company_key
WHERE f.salary_max_lpa IS NOT NULL
GROUP BY c.company_name
HAVING COUNT(f.salary_max_lpa) >= 5
ORDER BY avg_salary DESC
LIMIT 20;
""", engine)

,company_name,disclosed_jobs,avg_salary
0,Fortune 500 IT MNC,8,52.50
1,Wize Careers Consultants,5,36.20
2,US MNC (analytics),61,35.86
3,Leading IT Software Company,8,34.00
4,India's Largest IT Service Provider,7,30.00
5,Sunovaa Tech,5,29.00
6,KPI Partners,19,28.68
7,Mphasis,6,28.33
8,Turing Global India,5,28.00
9,client of kaizen,20,27.55


In [9]:
pd.read_sql("""
SELECT
    c.company_name,
    COUNT(f.salary_max_lpa) AS disclosed_jobs,
    MIN(f.salary_max_lpa) AS minimum_salary,
    MAX(f.salary_max_lpa) AS maximum_salary,
    ROUND(AVG(f.salary_max_lpa), 2) AS avg_salary
FROM fact_jobs f
JOIN dim_company c
    ON f.company_key = c.company_key
WHERE c.company_name IN ('Bean Hr Consulting', 'Insurtech')
  AND f.salary_max_lpa IS NOT NULL
GROUP BY c.company_name;
""", engine)

,company_name,disclosed_jobs,minimum_salary,maximum_salary,avg_salary
0,Bean Hr Consulting,14,0.0,90.0,12.86


In [11]:
pd.read_sql("""
SELECT
    c.company_name,
    f.job_id,
    f.salary_min_lpa,
    f.salary_max_lpa,
    f.salary_midpoint_lpa
FROM fact_jobs f
JOIN dim_company c
    ON f.company_key = c.company_key
WHERE c.company_name IN ('Bean Hr Consulting', 'Insurtech')
ORDER BY c.company_name, f.salary_max_lpa DESC;
""", engine)

,company_name,job_id,salary_min_lpa,salary_max_lpa,salary_midpoint_lpa
0,Bean Hr Consulting,1235,85.0,90.0,87.5
1,Bean Hr Consulting,3155,85.0,90.0,87.5
2,Bean Hr Consulting,20475,0.0,0.0,0.0
3,Bean Hr Consulting,19287,0.0,0.0,0.0
4,Bean Hr Consulting,2489,0.0,0.0,0.0
5,Bean Hr Consulting,4216,0.0,0.0,0.0
6,Bean Hr Consulting,4651,0.0,0.0,0.0
7,Bean Hr Consulting,19278,0.0,0.0,0.0
8,Bean Hr Consulting,17349,0.0,0.0,0.0
9,Bean Hr Consulting,19122,0.0,0.0,0.0


In [13]:
pd.read_sql("""
SELECT
    c.company_name,
    COUNT(f.salary_max_lpa) AS total_disclosed_rows,
    SUM(CASE WHEN f.salary_max_lpa > 0 THEN 1 ELSE 0 END) AS actual_disclosed_jobs,
    ROUND(
        AVG(NULLIF(f.salary_max_lpa, 0)), 
        2
    ) AS avg_disclosed_salary
FROM fact_jobs f
JOIN dim_company c
    ON f.company_key = c.company_key
WHERE c.company_name IN ('Bean Hr Consulting', 'Insurtech')
GROUP BY c.company_name;
""", engine)

,company_name,total_disclosed_rows,actual_disclosed_jobs,avg_disclosed_salary
0,Bean Hr Consulting,14,2.0,90.0


## Solving unique company discrepancy

In [7]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_jobs,

    COUNT(DISTINCT company_key) AS total_companies,

    COUNT(DISTINCT location_key) AS total_locations,

    ROUND(
        AVG(NULLIF(salary_midpoint_lpa, 0)),
        2
    ) AS avg_disclosed_salary,

    ROUND(
        AVG(NULLIF(salary_midpoint_lpa, 0)),
        2
    ) AS avg_salary_check,

    SUM(
        CASE
            WHEN salary_midpoint_lpa > 0 THEN 1
            ELSE 0
        END
    ) AS disclosed_jobs

FROM fact_jobs;
""", engine)

,total_jobs,total_companies,total_locations,avg_disclosed_salary,avg_salary_check,disclosed_jobs
0,23201,7022,1563,15.33,15.33,2766.0


In [9]:
pd.read_sql("""
SELECT
    salary_midpoint_lpa
FROM fact_jobs
WHERE salary_midpoint_lpa > 0
ORDER BY salary_midpoint_lpa;
""", engine)

,salary_midpoint_lpa
0,0.05
1,0.05
2,0.05
3,0.05
4,0.05
...,...
2761,75.00
2762,80.00
2763,87.50
2764,87.50


In [12]:
pd.read_sql("""
SELECT COUNT(*) AS companies_in_dimension
FROM dim_company;
""", engine)

,companies_in_dimension
0,7022


In [14]:
pd.read_sql("""
SELECT COUNT(DISTINCT company_key) AS companies_in_fact
FROM fact_jobs;
""", engine)

,companies_in_fact
0,7022


In [16]:
pd.read_sql("""
SELECT COUNT(DISTINCT f.company_key) AS orphan_company_keys
FROM fact_jobs f
LEFT JOIN dim_company c
    ON f.company_key = c.company_key
WHERE c.company_key IS NULL;
""", engine)

,orphan_company_keys
0,0


In [18]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_company_records,
    COUNT(DISTINCT company_name) AS unique_company_names
FROM dim_company;
""", engine)

,total_company_records,unique_company_names
0,7022,6944


## Data Warehouse & Power BI Validation

Before finalizing the Power BI dashboard, the data warehouse was validated against the dashboard outputs to ensure consistency and identify any discrepancies.

### Validation Checks Performed

The following checks were performed:

- **Total Jobs:** Verified that the fact table contains **23,201 jobs**.
- **Disclosed Jobs:** Identified **2,766 jobs** with valid salary midpoint values.
- **Average Disclosed Salary:** SQL result of **₹15.33 LPA**, matching the Power BI dashboard.
- **Salary Disclosure Rate:** Approximately **11.93%**, matching the Power BI dashboard.
- **Company Dimension Integrity:** `dim_company` contains **7,022 company records**.
- **Fact-Dimension Consistency:** `fact_jobs` contains **7,022 distinct company keys**.
- **Referential Integrity:** No orphan company keys were found when validating `fact_jobs` against `dim_company`.
- **Location Validation:** The warehouse contains **1,563 unique location records**, while the dashboard reports **198 unique primary cities**. These represent different levels of location granularity and are therefore not expected to be equal.
- **Salary Median:** The disclosed salary values were sorted and prepared for median validation against Power BI.

### Result

The core SQL calculations and Power BI dashboard metrics were successfully reconciled. The validation confirms that the **fact table and dimension tables are correctly connected and that the key dashboard metrics are consistent with the underlying data warehouse**.

This validation step increases confidence in the reliability of the final **India Job Market Intelligence Dashboard** and ensures that the analytical results presented in Power BI are supported by the SQL data warehouse.